# Análise de Frequência de Palavras — America's AI Action Plan

Este notebook lê o `AMERICA_AI_ACTION_PLAN_texto.txt` (texto integral extraído
do documento *America's AI Action Plan*, The White House, julho de 2025 —
excluindo cabeçalhos de página, numeração e notas de rodapé) e levanta:

1. As **palavras isoladas** mais frequentes no documento (unigramas);
2. Um ranking **combinado** de palavras isoladas e expressões compostas
   (bigramas), como *"artificial intelligence"* e *"national security"*.

## Importação e carregamento dos dados

In [ ]:
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt

txt_path = Path("AMERICA_AI_ACTION_PLAN_texto.txt")
full_text = txt_path.read_text(encoding="utf-8")

print(f"Total de caracteres: {len(full_text)}")

## Tokenização e remoção de stopwords

In [ ]:
STOPWORDS = {
    "the", "a", "an", "and", "of", "to", "in", "that", "for", "as", "on",
    "with", "by", "or", "be", "is", "are", "we", "our", "their", "all",
    "at", "from", "this", "these", "those", "such", "which", "it", "its",
    "within", "while", "also", "both", "other", "than", "then", "so",
    "but", "not", "no", "do", "does", "did", "has", "have", "had", "was",
    "were", "been", "being", "will", "would", "should", "could", "can",
    "may", "might", "must", "shall", "us", "any", "into", "across", "over",
    "under", "about", "between", "including", "particularly", "especially",
    "towards", "toward", "through", "among", "who", "what", "how", "if",
    "each", "more", "most", "some", "own", "where", "when", "because",
    "during", "before", "after", "without", "per", "there", "here",
}

# Tokens em sequência (mantendo a ordem original do texto, para permitir bigramas fiéis)
tokens = re.findall(r"[a-zA-Z]+(?:-[a-zA-Z]+)*", full_text.lower())
print(f"Total de tokens: {len(tokens)}")

# --- Unigramas (palavras isoladas) ---
# len(t) >= 2 (em vez de > 2) para preservar siglas curtas relevantes, como "ai" (AI)
unigram_tokens = [t for t in tokens if t not in STOPWORDS and len(t) >= 2]
unigram_counts = Counter(unigram_tokens)

# --- Bigramas (palavras conjuntas), preservando adjacência real no texto ---
bigram_counts = Counter()
for w1, w2 in zip(tokens, tokens[1:]):
    if w1 not in STOPWORDS and w2 not in STOPWORDS and len(w1) >= 2 and len(w2) >= 2:
        bigram_counts[f"{w1} {w2}"] += 1

print("Top 10 unigramas:", unigram_counts.most_common(10))
print("Top 10 bigramas:", bigram_counts.most_common(10))

## Gráfico 1 — Palavras isoladas mais frequentes

Ranking das 30 palavras (unigramas) que mais aparecem no documento.

In [ ]:
SEQUENTIAL_BLUE = "#2a78d6"
INK_PRIMARY = "#0b0b0b"
INK_MUTED = "#898781"
GRIDLINE = "#e1e0d9"
SURFACE = "#fcfcfb"

def plot_ranked_bar(pairs, title, filename=None):
    labels = [p[0] for p in pairs][::-1]
    values = [p[1] for p in pairs][::-1]

    fig, ax = plt.subplots(figsize=(8, 8), facecolor=SURFACE)
    ax.set_facecolor(SURFACE)

    bars = ax.barh(labels, values, color=SEQUENTIAL_BLUE, height=0.62, zorder=3)

    ax.set_xlabel("Frequência (nº de ocorrências)", color=INK_MUTED, fontsize=10)
    ax.set_title(title, color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)

    ax.tick_params(axis="y", colors=INK_PRIMARY, labelsize=10)
    ax.tick_params(axis="x", colors=INK_MUTED, labelsize=9)

    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color(GRIDLINE)

    ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)

    max_val = max(values)
    for bar, value in zip(bars, values):
        ax.text(
            bar.get_width() + max_val * 0.015,
            bar.get_y() + bar.get_height() / 2,
            str(value),
            va="center", ha="left",
            color=INK_PRIMARY, fontsize=9,
        )
    ax.set_xlim(0, max_val * 1.12)

    fig.tight_layout()
    if filename:
        fig.savefig(filename, dpi=150, facecolor=SURFACE, bbox_inches="tight")
    plt.show()

top_unigrams = unigram_counts.most_common(30)
plot_ranked_bar(
    top_unigrams,
    "Palavras isoladas mais frequentes — America's AI Action Plan",
)

## Gráfico 2 — Palavras isoladas e expressões conjuntas (bigramas)

Combina as 15 palavras isoladas mais frequentes com as 15 expressões
compostas (bigramas) mais frequentes — como *"artificial intelligence"*,
*"national security"* e *"federal government"* — em um único ranking.

In [ ]:
TOP_N = 15

# Um puro ranking por frequência bruta esconde os bigramas (naturalmente mais raros
# que palavras isoladas). Por isso, o gráfico combina explicitamente os N unigramas
# e os N bigramas mais frequentes, garantindo que expressões como
# "artificial intelligence" apareçam lado a lado com palavras isoladas como
# "innovation".
top_unigrams_for_combo = unigram_counts.most_common(TOP_N)
top_bigrams_for_combo = bigram_counts.most_common(TOP_N)

combined_pairs = top_unigrams_for_combo + top_bigrams_for_combo
combined_pairs.sort(key=lambda x: x[1], reverse=True)

plot_ranked_bar(
    combined_pairs,
    "Palavras isoladas e expressões conjuntas mais frequentes — America's AI Action Plan",
)